# 01 — Data Explorer

Purpose: one cell per data source, showing role, resolution, revisit, latency, and the
governing decision record. Every source below is **simulated** — there are no API
credentials configured yet (`.env` is a placeholder; see `docs/decisions/data-access-layer.md`
and `docs/decisions/weather-forcing-split.md`).

Pilot AOI (placeholder, not final — see `docs/open-decisions.md` target cropping pattern):
Aegean region, İzmir/Manisa, bounding box lat 38.4–38.7N, lon 27.0–27.5E.

This notebook does not import anything from `packages/`. Per `CLAUDE.md`, production code
must never import from `experiments/`, and this notebook is throwaway/exploratory.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

# Placeholder pilot AOI — Aegean (İzmir/Manisa). Not the final target cropping pattern.
AOI_BBOX = dict(lat_min=38.4, lat_max=38.7, lon_min=27.0, lon_max=27.5)
WINDOW_DAYS = 30
dates = pd.date_range("2025-06-01", periods=WINDOW_DAYS, freq="D", tz="UTC")

def seasonal_signal(n, base, amplitude, noise_std, period=365, phase=150, rng=rng):
    t = np.arange(n)
    signal = base + amplitude * np.sin(2 * np.pi * (t + phase) / period)
    return signal + rng.normal(0, noise_std, size=n)


Matplotlib is building the font cache; this may take a moment.


## Sentinel-1 RTC (SAR)

- **Role:** primary backscatter signal for soil-moisture-sensitive Layer B (state estimation)
  and Layer A (observation gating) in `docs/decisions/ml-layering.md`.
- **Resolution:** ~10-20 m (pre-built RTC gamma0 product).
- **Revisit:** ~6 days (single satellite; better with constellation coverage).
- **Latency:** hours to ~1 day for RTC products from the provider.
- **Governs:** `docs/decisions/data-access-layer.md` — pre-built RTC only, no raw GRD
  processing. Terrain flattening and layover/shadow masking are mandatory for the Aegean's
  uneven terrain.

**SIMULATED DATA — replace with live fetch once credentials are configured in .env**


In [2]:
# Simulated Sentinel-1 gamma0 backscatter (VV), dB, with a moisture-driven seasonal component
s1_vv_db = seasonal_signal(WINDOW_DAYS, base=-11.0, amplitude=1.5, noise_std=0.4)
s1_layover_shadow_mask = rng.random(WINDOW_DAYS) < 0.05  # ~5% geometrically invalid, per data-access-layer.md

s1_df = pd.DataFrame({"date": dates, "gamma0_vv_db": s1_vv_db, "layover_shadow": s1_layover_shadow_mask})

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(s1_df["date"], s1_df["gamma0_vv_db"], marker="o", ms=3)
ax.set_title("Simulated Sentinel-1 RTC gamma0 (VV) — placeholder AOI")
ax.set_ylabel("dB")
plt.tight_layout()
plt.show()


/var/folders/0s/rggf_js519j8hsgy53fr_jx80000gn/T/ipykernel_76133/1506972310.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Layer A connection:** every S1 acquisition above would be run through the observation
gating classifier (`ml-layering.md` Layer A) before assimilation — usability class
(soil-dominated / vegetation-dominated / wet canopy / frozen / roughness-changed /
geometrically invalid) sets the EnKF observation error covariance `R`, not a hard accept/reject
mask. `layover_shadow=True` rows above are the geometrically-invalid case.


## Sentinel-2 L2A (Optical)

- **Role:** NDVI / vegetation correction for the WCM forward operator; optional Kc refinement
  in Phase 2 (`docs/delivery-plan.md`).
- **Resolution:** 10 m (visible/NIR bands).
- **Revisit:** ~5 days (single satellite), degraded by cloud cover in practice.
- **Latency:** hours for L2A once processed.
- **Governs:** `docs/decisions/data-access-layer.md` — accessed via Copernicus Data Space
  (openEO / Sentinel Hub).

**SIMULATED DATA — replace with live fetch once credentials are configured in .env**


In [3]:
# Simulated NDVI with a growing-season ramp and simulated cloud gaps (NaN)
ndvi = seasonal_signal(WINDOW_DAYS, base=0.55, amplitude=0.15, noise_std=0.02)
cloud_mask = rng.random(WINDOW_DAYS) < 0.35  # ~35% cloud-obscured scenes, typical for the region
ndvi_observed = np.where(cloud_mask, np.nan, ndvi)

s2_df = pd.DataFrame({"date": dates, "ndvi": ndvi_observed, "cloud_masked": cloud_mask})

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(s2_df["date"], s2_df["ndvi"], marker="o", ms=3, label="observed (cloud gaps as NaN)")
ax.set_title("Simulated Sentinel-2 NDVI — placeholder AOI")
ax.set_ylabel("NDVI")
ax.legend()
plt.tight_layout()
plt.show()
print(f"Simulated cloud-gap rate: {cloud_mask.mean():.0%}")


Simulated cloud-gap rate: 33%


/var/folders/0s/rggf_js519j8hsgy53fr_jx80000gn/T/ipykernel_76133/3392861224.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Landsat-8/9 (Optical)

- **Role:** long-horizon historical reference and cross-calibration source; not yet formally
  adopted in a decision record. **Note:** `docs/decisions/data-access-layer.md` currently only
  names Sentinel-1/2, weather, and SoilGrids — a Landsat decision record is a documented gap,
  see the summary note at the end of this notebook.
- **Resolution:** 30 m (visible/NIR/thermal).
- **Revisit:** ~16 days (single satellite; ~8 days combined Landsat-8+9).
- **Latency:** hours to ~1 day via USGS/Microsoft Planetary Computer STAC catalogs.

**SIMULATED DATA — replace with live fetch once credentials are configured in .env**


In [4]:
# Landsat revisit is coarser — simulate every ~8th day only (8-day combined revisit)
landsat_dates = dates[::8]
landsat_ndvi = seasonal_signal(len(landsat_dates), base=0.53, amplitude=0.15, noise_std=0.03)

landsat_df = pd.DataFrame({"date": landsat_dates, "ndvi": landsat_ndvi})

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(s2_df["date"], s2_df["ndvi"], marker="o", ms=3, alpha=0.4, label="S2 (dense, cloud gaps)")
ax.plot(landsat_df["date"], landsat_df["ndvi"], marker="s", ms=5, label="Landsat (sparse, ~8-day)")
ax.set_title("Simulated NDVI: Sentinel-2 vs Landsat revisit density")
ax.legend()
plt.tight_layout()
plt.show()


/var/folders/0s/rggf_js519j8hsgy53fr_jx80000gn/T/ipykernel_76133/3998001894.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## SMAP (Soil Moisture Active Passive, NASA)

- **Role:** primary satellite-derived soil moisture reference — used to cross-validate the
  twin's `theta(z,t)` estimate and, per the user's explicit request, cross-validated against
  ISMN ground stations (see notebook 02 and 04).
- **Resolution:** ~9 km (enhanced product) — much coarser than S1/S2, so it is a
  regional-consistency check, not a zone-level input.
- **Revisit:** ~2-3 days (daily composite product).
- **Latency:** ~2-3 days for near-real-time product.
- **Governs:** no dedicated decision record yet — flagged as a gap below.

**SIMULATED DATA — replace with live fetch once credentials are configured in .env**


In [5]:
# Simulated SMAP surface soil moisture, m3/m3, seasonal + noise, clipped to physical range
smap_theta = seasonal_signal(WINDOW_DAYS, base=0.22, amplitude=0.08, noise_std=0.015)
smap_theta = np.clip(smap_theta, 0.0, 1.0)  # CLAUDE.md rule 5: soil moisture is always m3/m3, physically in [0, 1]

smap_df = pd.DataFrame({"date": dates, "theta_m3m3": smap_theta})

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(smap_df["date"], smap_df["theta_m3m3"], marker="o", ms=3, color="tab:blue")
ax.set_title("Simulated SMAP surface soil moisture — placeholder AOI")
ax.set_ylabel("theta (m3/m3)")
plt.tight_layout()
plt.show()


/var/folders/0s/rggf_js519j8hsgy53fr_jx80000gn/T/ipykernel_76133/248141290.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## ISMN (International Soil Moisture Network — ground stations)

- **Role:** independent, point-scale ground-truth used to cross-validate SMAP (both pipelines
  run in parallel and are compared, not merged into one series — see notebooks 02 and 04).
- **Resolution:** point measurement (station footprint, effectively meters).
- **Revisit:** sub-daily to daily depending on station.
- **Latency:** varies by network operator; often days to weeks for QC'd data.
- **Governs:** no dedicated decision record yet — flagged as a gap below.

**SIMULATED DATA — replace with live fetch once credentials are configured in .env**


In [6]:
# Simulated ISMN point soil moisture — same underlying process as SMAP plus a small point-vs-footprint
# bias and independent station noise, so the two series are correlated but not identical.
ismn_theta = smap_theta + rng.normal(0.01, 0.02, size=WINDOW_DAYS)  # small wet bias + station noise
ismn_theta = np.clip(ismn_theta, 0.0, 1.0)

ismn_df = pd.DataFrame({"date": dates, "theta_m3m3": ismn_theta})

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(smap_df["date"], smap_df["theta_m3m3"], marker="o", ms=3, label="SMAP (9 km footprint)")
ax.plot(ismn_df["date"], ismn_df["theta_m3m3"], marker="x", ms=5, label="ISMN (point station)")
ax.set_title("Simulated soil moisture: SMAP vs ISMN — two independent pipelines")
ax.set_ylabel("theta (m3/m3)")
ax.legend()
plt.tight_layout()
plt.show()


/var/folders/0s/rggf_js519j8hsgy53fr_jx80000gn/T/ipykernel_76133/2709439665.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## SoilGrids (static)

- **Role:** static soil texture/property inputs to zone delineation (k-means over NDVI +
  SoilGrids, per `docs/decisions/spatial-analysis-unit.md`) and to the twin's soil physics
  priors.
- **Resolution:** 250 m.
- **Revisit:** static — one-off download, no time dimension.
- **Latency:** n/a (downloaded once).
- **Governs:** `docs/decisions/data-access-layer.md` — static one-off download.

**SIMULATED DATA — replace with live fetch once credentials are configured in .env**


In [7]:
# Simulated static SoilGrids properties for a handful of candidate sub-parcel points
n_points = 20
soilgrids_df = pd.DataFrame({
    "point_id": range(n_points),
    "clay_pct": rng.normal(28, 6, n_points).clip(0, 100),
    "sand_pct": rng.normal(40, 8, n_points).clip(0, 100),
    "bulk_density_kg_m3": rng.normal(1350, 60, n_points),
})
soilgrids_df.head()


,point_id,clay_pct,sand_pct,bulk_density_kg_m3
0,0,25.534437,40.724679,1360.594402
1,1,34.637732,45.151510,1284.936716
2,2,30.572539,23.598623,1355.429387
3,3,37.214536,39.610253,1363.693700
4,4,29.099407,33.254158,1501.048442


## Open-Meteo (Operational weather)

- **Role:** operational forcing line for daily inference and recipe generation (ET0 via
  Penman-Monteith, see notebook 02).
- **Resolution:** point/grid forecast, varies by underlying model (typically ~10-25 km).
- **Revisit:** hourly.
- **Latency:** near-real-time (hours or better).
- **Governs:** `docs/decisions/weather-forcing-split.md` — operational line, explicitly
  separate from the archive (ERA5-Land) line. Importing ERA5-Land into the operational path is
  prohibited and checked in code review.

**SIMULATED DATA — replace with live fetch once credentials are configured in .env**


In [8]:
# Simulated Open-Meteo daily weather (operational line)
open_meteo_df = pd.DataFrame({
    "date": dates,
    "t_mean_c": seasonal_signal(WINDOW_DAYS, base=24, amplitude=4, noise_std=1.0),
    "t_max_c": seasonal_signal(WINDOW_DAYS, base=30, amplitude=4, noise_std=1.2),
    "t_min_c": seasonal_signal(WINDOW_DAYS, base=18, amplitude=3, noise_std=1.0),
    "rh_mean_pct": seasonal_signal(WINDOW_DAYS, base=55, amplitude=10, noise_std=3).clip(0, 100),
    "wind_2m_ms": np.abs(rng.normal(2.2, 0.6, WINDOW_DAYS)),
    "solar_rad_mj_m2_day": seasonal_signal(WINDOW_DAYS, base=24, amplitude=4, noise_std=1.5).clip(0, None),
    "forcing_line": "operational",
})
open_meteo_df.head()


,date,t_mean_c,t_max_c,t_min_c,rh_mean_pct,wind_2m_ms,solar_rad_mj_m2_day,forcing_line
0,2025-06-01 00:00:00+00:00,24.365192,32.006176,20.456018,58.210509,2.799895,25.415256,operational
1,2025-06-02 00:00:00+00:00,24.597204,34.576052,19.219662,57.119472,1.564878,26.753328,operational
2,2025-06-03 00:00:00+00:00,28.134214,33.892993,19.442401,60.110764,2.124995,27.057898,operational
3,2025-06-04 00:00:00+00:00,24.657668,32.408107,18.405920,56.213047,3.088873,26.152453,operational
4,2025-06-05 00:00:00+00:00,24.787852,30.968969,19.079022,57.698174,1.753847,27.024838,operational


## ERA5-Land (Archive weather)

- **Role:** training and retrospective calibration line only. **Never used for operational
  inference** — publication lag is ~2-3 months.
- **Resolution:** ~9 km (0.1°).
- **Revisit:** hourly, back to 2017 (consistent archive).
- **Latency:** ~2-3 months.
- **Governs:** `docs/decisions/weather-forcing-split.md` — archive line, kept in strictly
  separate columns from the operational line, never silently merged (see notebook 03).

**SIMULATED DATA — replace with live fetch once credentials are configured in .env**


In [9]:
# Simulated ERA5-Land daily weather (archive line) — same physical process, independent noise
# realization, to stand in for the fact these are two genuinely different data products.
era5_df = pd.DataFrame({
    "date": dates,
    "t_mean_c": seasonal_signal(WINDOW_DAYS, base=24, amplitude=4, noise_std=0.8),
    "t_max_c": seasonal_signal(WINDOW_DAYS, base=30, amplitude=4, noise_std=1.0),
    "t_min_c": seasonal_signal(WINDOW_DAYS, base=18, amplitude=3, noise_std=0.8),
    "rh_mean_pct": seasonal_signal(WINDOW_DAYS, base=55, amplitude=10, noise_std=2.5).clip(0, 100),
    "wind_2m_ms": np.abs(rng.normal(2.1, 0.5, WINDOW_DAYS)),
    "solar_rad_mj_m2_day": seasonal_signal(WINDOW_DAYS, base=24, amplitude=4, noise_std=1.2).clip(0, None),
    "forcing_line": "archive",
})
era5_df.head()


,date,t_mean_c,t_max_c,t_min_c,rh_mean_pct,wind_2m_ms,solar_rad_mj_m2_day,forcing_line
0,2025-06-01 00:00:00+00:00,25.647400,34.073933,19.665406,60.555994,2.223833,26.437363,archive
1,2025-06-02 00:00:00+00:00,24.879820,34.141230,20.012809,59.945372,2.010387,24.984616,archive
2,2025-06-03 00:00:00+00:00,25.294460,32.074348,19.458299,61.989433,1.973311,26.232779,archive
3,2025-06-04 00:00:00+00:00,25.658677,32.105281,19.322492,60.724340,2.020408,24.199304,archive
4,2025-06-05 00:00:00+00:00,26.527506,32.960878,18.789893,61.382410,2.201694,27.488061,archive


## Summary: documented gaps found in this pass

While grounding this notebook in the decision records, two gaps surfaced:

1. **Landsat-8/9 has no decision record.** `docs/decisions/data-access-layer.md` only covers
   Sentinel-1, Sentinel-2, weather, and SoilGrids. Landsat is used here as a placeholder
   long-horizon reference source. Per `CLAUDE.md` rule 1, a decision record should be proposed
   before Landsat is wired into production code.
2. **SMAP and ISMN have no decision record.** They are the user-requested dual soil-moisture
   pipeline (SMAP primary, ISMN cross-validation), but neither appears in
   `docs/decisions/data-access-layer.md` or elsewhere. This should be written up as a decision
   record (or an addendum to `data-access-layer.md`) before production ingestion code is built.

These are flagged, not resolved, here — resolving them is a decision-record task, not a
notebook task.
